# 18 - רשת משוקללת בזמני נסיעה (Travel-Time Weighted Network)

כל המחברות בפרויקט זה עד כה מדדו מרחק ב**קפיצות** (hops): המסלול הקצר ביותר בין שתי תחנות הוא זה
שעובר דרך מספר התחנות המזערי. זוהי מגבלה אמיתית שברובה לא נאמרה במפורש. שתי תחנות אוטובוס עירוניות
עוקבות במרחק 200 מ' זו מזו נספרות בדיוק כמו קטע רכבת בין-עירוני בן 40 דקות, ולכן מסלול "קצר" בגרף
הקפיצות עשוי להיות מסע ארוך, ותחנה הנראית כצוואר בקבוק תחת ספירת קפיצות עשויה להיות חסרת חשיבות
ברגע שהדקות הן שקובעות.

מחברת זו מסירה את המגבלה הזו. היא עוברת פעם אחת בזרימה על קובץ ה-feed הגולמי `stop_times.txt`,
ולכל זוג תחנות עוקבות בתוך trip מחשבת את זמן הנסיעה המתוכנן בשניות. איסוף ההתפלגות המלאה לכל קטע
מניב **זמן נסיעה median / p25 / p75 לכל קשת**, אשר הופך למשקל הקשת בגרף חדש, `graph_traveltime.pkl`.
גרף זה הוא הבסיס למחברות 20-22 (חוסן דינמי, קריטיות משוקללת בביקוש, עלות ניתוב מחדש), שאף אחת מהן
אינה יכולה להפיק מספר בעל משמעות של "עיקוף בדקות" על גרף קפיצות.

לאחר מכן שואלת המחברת את השאלה שקובעת כמה מן העבודה הקודמת חייבת להיקרא מחדש:
**האם השקלול מחדש משנה מי הן התחנות הקריטיות?** אנו משווים מסלולים קצרים ביותר לפי זמן נסיעה
למסלולים קצרים ביותר לפי ספירת קפיצות על מדגם של זוגות מוצא-יעד, מחשבים מחדש betweenness על גרף
זמני הנסיעה, ומתאמים אותו עם ה-betweenness מבוסס הקפיצות ממחברת 04. אם הדירוג זז באופן מהותי, זו
הסתייגות על כל טענה מבוססת-מרכזיות שנטענה עד כה, ואנו אומרים זאת.

**שאלת המחקר:** כמה מן המבנה הנמדד של הרשת הוא תוצר לוואי (artefact) של שימוש בספירת קפיצות כמרחק,
ואילו תחנות מרוויחות או מאבדות חשיבות ברגע שהקשתות משוקללות בדקות מתוכננות?

## קלט

| נתיב | הופק על ידי | משמש עבור |
|---|---|---|
| `israel-public-transportation/stop_times.txt` | GTFS גולמי (816 MB, **לא נמצא ב-git** - מורד בהמשך) | תצפיות זמני הנסיעה |
| `outputs/nb/02_graph_construction/tables/nodes.csv` | מחברת **02** | שמות תחנות, קואורדינטות, מחוז, מטרופולין |
| `outputs/nb/02_graph_construction/tables/edges.csv` | מחברת **02** | `trip_frequency` לכל קטע מכוון |
| `outputs/nb/04_centrality_analysis/tables/stop_metrics.csv` | מחברת **04** | `approx_betweenness` - קו הבסיס מבוסס הקפיצות |

## יש להריץ קודם

`01_data_preparation` -> `02_graph_construction` -> `04_centrality_analysis`.
כל טוען (loader) בהמשך מעלה `FileNotFoundError` מפורש המציין את שם המחברת שיש להריץ אם חסר artifact.

## פלט (הכל תחת `outputs/nb/18_travel_time_network/`)

| נתיב | תוכן |
|---|---|
| `tables/edges_traveltime.csv` | `from_stop, to_stop, median_travel_seconds, p25_travel_seconds, p75_travel_seconds, trip_frequency, n_observations` - **טבלת החוזה שמחברות אחרות קוראות** |
| `graph_traveltime.pkl` | `networkx.Graph` לא-מכוון; תכונת קשת `travel_seconds` (משוקפת גם ל-`weight`) |
| `traveltime_summary.json` | מספרי הכותרת, כל מוני הפסילות, והשוואת ה-betweenness |
| `tables/discard_reasons.csv` | כמה תצפיות קטע נפסלו ומדוע |
| `tables/path_comparison.csv` | השוואה לכל זוג בין המסלול הקצר בקפיצות למסלול הקצר בזמן |
| `tables/betweenness_traveltime.csv` | betweenness לפי זמן נסיעה לצד שני קווי הבסיס מבוססי הקפיצות |
| `tables/betweenness_rank_movers.csv` | התחנות שדירוג ה-betweenness שלהן זז הכי הרבה |
| `figures/*.png` | חמישה איורים דיאגנוסטיים |

דבר מחוץ ל-`outputs/nb/18_travel_time_network/` אינו נכתב. התיקיות המצוטטות בדוח
`outputs/tables`, `outputs/figures` ו-`outputs/rail` אינן נוגעות כלל.

## 1. אתחול סביבת העבודה

התא שלהלן הופך את המחברת לניתנת להרצה הן על עותק מקומי של הריפוזיטורי והן על Google Colab. הוא מגדיר
את `_ensure(...)`, המתקין באמצעות pip רק את החבילות שחסרות באמת (כך שהרצה חוזרת של המחברת זולה), ואת
`find_repo_root()`, המטפס כלפי מעלה מהתיקייה הנוכחית בחיפוש אחר תיקיית ה-GTFS, ואם אינו מוצא אותה
משכפל (clone) את הריפוזיטורי אל `/content`. לאחר מכן הוא מגדיר את `REPO`, `DATA` ו-`OUT` ויוצר את
שורש הפלט של המחברת. כל תא מאוחר יותר מסתמך על שלושת הנתיבים הללו, ולכן תא זה חייב לרוץ ראשון.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. ספריות, תיקיות שלב וקבועי עלות

מחסנית מדעית סטנדרטית בתוספת המודול `csv` מספריית התקן, שהוא זה שקורא בפועל את ה-feed בנפח 816 MB
שורה אחר שורה. תיקיית השלב עוקבת אחר מוסכמת הפרויקט: מחברת זו מחזיקה בבעלותה את
`outputs/nb/18_travel_time_network/` ואינה כותבת לשום מקום אחר.

כל פרמטרי זמן הריצה וכל ספי המידול מרוכזים כאן כדי שבודק יוכל לשנותם במקום אחד. היקרים שבהם, עם
הערכות עלות כנות על גרף בן 30.5k צמתים / 51.8k קשתות:

| קבוע | ברירת מחדל | עלות |
|---|---|---|
| מעבר הזרימה על `stop_times.txt` | תמיד רץ | **3-6 דקות**, שיא של כ-150 MB RAM עבור מערכי משכי הזמן לכל קטע |
| `RUN_BETWEENNESS` | `True` | קבעו ל-`False` כדי לדלג לחלוטין על סעיפים 9-11 |
| `K_BETWEENNESS` | `200` מקורות שנדגמו | betweenness **משוקלל** דורש Dijkstra לכל מקור במקום BFS: יש לתקצב **5-15 דקות** להרצה לפי זמני נסיעה בתוספת **1-3 דקות** להרצת הביקורת מבוססת הקפיצות |
| `N_PATH_PAIRS` | `300` זוגות מוצא-יעד | כ-1-3 דקות (BFS דו-כיווני אחד ו-Dijkstra דו-כיווני אחד לכל זוג) |

ספי המידול הם `MIN_TRAVEL_SECONDS` ו-`MAX_TRAVEL_SECONDS`: משך קטע נצפה מחוץ לתחום
`[1 s, 3 h]` נפסל כשגיאת נתונים ולא זוכה לאמון. סעיף 6 מדווח בדיוק כמה תצפיות הסיר כל כלל, כך
שהבחירה ניתנת לביקורת ואינה מוסתרת.

In [ ]:
_ensure("pandas", "numpy", "networkx", "matplotlib", "seaborn")

import csv, json, pickle, random, time
from array import array
from collections import defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.05)
csv.field_size_limit(10_000_000)   # a few rows in the feed are unusually long

# ---------------- stage folders ----------------
STAGE = OUT / "18_travel_time_network"
TABLES = STAGE / "tables"
FIGURES = STAGE / "figures"
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

# ---------------- modelling thresholds ----------------
MIN_TRAVEL_SECONDS = 1        # a segment must take strictly positive time
MAX_TRAVEL_SECONDS = 3 * 3600 # 3 hours: above this it is a feed error, not a bus ride
MIN_OBSERVATIONS = 1          # segments with fewer valid observations are dropped

# ---------------- cost knobs (see the table above) ----------------
PROGRESS_EVERY = 2_000_000    # progress print interval during the streaming pass
RUN_BETWEENNESS = True        # False -> skip the (slow) betweenness sections 9-11
K_BETWEENNESS = 200           # sampled sources for approximate betweenness
BETWEENNESS_SEED = 42         # same seed for both runs => same sampled source set
N_PATH_PAIRS = 300            # origin-destination pairs in the path comparison
PATH_SEED = 11
RANK_POOL = 500               # rank-mover analysis is restricted to this top-N pool
TOP_N = 15
FIG_DPI = 150

print("pandas", pd.__version__, "| networkx", nx.__version__)
print("stage folder:", STAGE)

## 3. עיבוד (rendering) תוויות בעברית

שמות התחנות ב-feed ה-GTFS הישראלי הם בעברית, ומספר איורים בהמשך מדפיסים אותם. Matplotlib אינו מממש
את אלגוריתם ה-bidirectional של Unicode, ולכן טקסט מימין לשמאל יוצא הפוך ובלתי קריא. התא שלהלן מבצע
monkey-patch חד-פעמי ל-`matplotlib.text.Text.set_text` כך שכל מחרוזת המכילה תווים עבריים מומרת לסדר
תצוגה באמצעות `python-bidi` לפני שהיא מצוירת, ובוחר גופן שיש בו באמת גליפים עבריים (Arial ב-Windows,
DejaVu Sans בכל שאר המערכות). הפעולה אידמפוטנטית - הרצה חוזרת שלה לא תערים patch-ים זה על זה. כל
שאר הטקסט במחברת הוא באנגלית.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. איתור שלבים קודמים

תיקיות השלבים מאותרות לפי **הקידומת הדו-ספרתית** שלהן (`OUT.glob("02*")`), ולא לפי slug מדויק, כך
ששינוי שם של תיקייה אינו שובר את השרשרת. `stage_file` מעלה `FileNotFoundError` המציין את שם המחברת
שיש להריץ תחילה, משום שנפילה חרישית לברירת מחדל כאן הייתה מייצרת גרף ללא שמות תחנות וללא קו בסיס
להשוואה - כשל שהיה מתגלה רק כעבור מספר סעיפים.

In [ ]:
def stage_dir(prefix, notebook_hint):
    """Resolve a stage folder by its two-digit prefix, e.g. '02' -> 02_graph_construction."""
    matches = sorted(p for p in OUT.glob(f"{prefix}*") if p.is_dir())
    if not matches:
        raise FileNotFoundError(
            f"No stage folder matching '{prefix}*' under {OUT} - "
            f"run notebook {notebook_hint} first."
        )
    return matches[0]


def stage_file(prefix, filename, notebook_hint):
    """Return the path of `filename` inside stage `prefix`, searching sub-folders."""
    root = stage_dir(prefix, notebook_hint)
    direct = root / filename
    if direct.exists():
        return direct
    hits = sorted(root.rglob(filename))
    if not hits:
        raise FileNotFoundError(
            f"'{filename}' not found anywhere under {root} - "
            f"run notebook {notebook_hint} first; it is the notebook that writes this file."
        )
    return hits[0]


NODES_CSV = stage_file("02", "nodes.csv", "02_graph_construction")
EDGES_CSV = stage_file("02", "edges.csv", "02_graph_construction")
METRICS_CSV = stage_file("04", "stop_metrics.csv", "04_centrality_analysis")

nodes_df = pd.read_csv(NODES_CSV, dtype={"stop_id": str}, encoding="utf-8-sig")
edges_df = pd.read_csv(EDGES_CSV, dtype={"from_stop": str, "to_stop": str}, encoding="utf-8-sig")

print(f"nodes.csv       : {len(nodes_df):,} rows  <- {NODES_CSV}")
print(f"edges.csv       : {len(edges_df):,} rows  <- {EDGES_CSV}")
print(f"stop_metrics.csv: found            <- {METRICS_CSV}")

ATTR = {
    str(r["stop_id"]): {
        "stop_name": ("" if pd.isna(r.get("stop_name")) else str(r.get("stop_name"))),
        "lat": (np.nan if pd.isna(r.get("lat")) else float(r.get("lat"))),
        "lon": (np.nan if pd.isna(r.get("lon")) else float(r.get("lon"))),
        "region": ("" if pd.isna(r.get("region")) else str(r.get("region"))),
        "metro": ("" if pd.isna(r.get("metro")) else str(r.get("metro"))),
    }
    for r in nodes_df.to_dict("records")
}
DEFAULT_ATTR = {"stop_name": "", "lat": np.nan, "lon": np.nan, "region": "", "metro": ""}

# Directed segment frequencies from stage 02, keyed exactly as edges.csv stores them.
FREQ = {(str(r["from_stop"]), str(r["to_stop"])): int(r["trip_frequency"])
        for r in edges_df.to_dict("records")}
print(f"directed segments with a known trip_frequency: {len(FREQ):,}")

## 5. פענוח זמני GTFS - מוסכמת ה->= 24:00

זהו הפרט המסוכן ביותר במחברת, ולכן הוא מקבל סעיף משלו ובדיקות משלו.

שדות השעון ב-GTFS **אינם** זמני שעון-קיר. Trip המתחיל ב-23:50 וממשיך אל מעבר לחצות נכתב כ-
`23:50:00`, `24:05:00`, `25:30:00`, ... - שדה השעה ממשיך לספור מעבר ל-23 כדי שה-trip כולו יישאר בתוך
*יום שירות* אחד. ב-feed זה כ-**1.1%** מהשורות נושאות שדה שעה של 24 ומעלה.

שתי השלכות:

1. **`datetime.strptime` אינו מסוגל לפענח ערכים אלה - הוא מעלה שגיאה.** ולכן איננו משתמשים בו לעולם.
2. אילו היינו מנרמלים את השעה מודולו 24 (`25:30` -> `01:30`) כדי לרצות את `strptime`, החיסור
   `arrival(v) - departure(u)` היה מחזיר חרישית **-22.3 שעות** במקום +7 דקות עבור כל קטע החוצה את
   חצות. זו אינה קריסה, זהו מספר שגוי, וזה גרוע יותר.

הייצוג הנכון הוא **שניות מאז חצות יום השירות**: `int(h)*3600 + int(m)*60 + int(s)`, כאשר ל-`h` מותר
לעבור את 23. הפרשים בין שני ערכים כאלה בתוך אותו trip הם אז תמיד נכונים, כולל בחציית חצות, משום
ששתי נקודות הקצה חיות על אותו ציר מונוטוני.

`gtfs_seconds` מחזיר `None` (לא קריסה, ולא אפס) עבור ערכים ריקים או פגומים, כך שזמנים חסרים נספרים
כסיבת פסילה במקום להפוך חרישית ל-00:00:00. ה-assertions שלהלן מקבעים את ההתנהגות - כולל מקרה חציית
חצות המדויק שהוא הסיבה לכל הסעיף הזה.

In [ ]:
def gtfs_seconds(value):
    """GTFS 'HH:MM:SS' -> seconds since *service* midnight. Hours >= 24 are legal.

    Returns None for blank or malformed input. Never uses datetime/strptime, which
    raises on '25:30:00' - see the markdown above.
    """
    if not value:
        return None
    s = value.strip()
    if not s:
        return None
    parts = s.split(":")
    if len(parts) != 3:
        return None
    try:
        h, m, sec = int(parts[0]), int(parts[1]), int(parts[2])
    except ValueError:
        return None
    return h * 3600 + m * 60 + sec


# --- lock the contract with tests -----------------------------------------
assert gtfs_seconds("00:00:00") == 0
assert gtfs_seconds("05:12:23") == 5 * 3600 + 12 * 60 + 23
assert gtfs_seconds("25:30:00") == 25 * 3600 + 30 * 60          # 91800, NOT 5400
assert gtfs_seconds(" 5:07:00") == 5 * 3600 + 7 * 60            # single-digit hour
assert gtfs_seconds("") is None and gtfs_seconds(None) is None
assert gtfs_seconds("not a time") is None

# The midnight-crossing case that a naive parser gets catastrophically wrong.
depart_u = gtfs_seconds("23:58:00")
arrive_v = gtfs_seconds("24:06:00")
assert arrive_v - depart_u == 8 * 60, "midnight-crossing segment must be +8 minutes"
naive_wrong = (gtfs_seconds("00:06:00") - depart_u) / 60.0
print(f"service-midnight parsing : {(arrive_v - depart_u) / 60:.0f} min  (correct)")
print(f"naive hour-mod-24 parsing: {naive_wrong:.0f} min  (silently wrong)")
print("gtfs_seconds passes all contract tests.")

## 6. תלות בנתונים חיצוניים: `stop_times.txt`

`stop_times.txt` שוקל 816 MB / 15.7M שורות - הרבה מעבר למגבלת גודל הקובץ של GitHub - ולכן הוא **אינו**
נמצא בריפוזיטורי. התא שלהלן מוריד אותו מ-Google Drive בהרצה הראשונה ומדלג על ההורדה אם הקובץ כבר
קיים. זוהי תלות הרשת החיצונית היחידה של המחברת. בהרצה ראשונה ב-Colab ההורדה נמשכת מספר דקות.

In [ ]:
# stop_times.txt is 816MB and is not tracked in git - fetch it on demand.
_ensure("gdown")
import gdown
STOP_TIMES = DATA / "stop_times.txt"
if not STOP_TIMES.exists():
    gdown.download(id="1V_yPAWXV6mGTFGrfiosah5LngcLZnviW",
                   output=str(STOP_TIMES), quiet=False)
print("stop_times.txt:", round(STOP_TIMES.stat().st_size / 1024**2, 1), "MB")

## 7. מעבר הזרימה: התפלגות משכים אחת לכל קטע

זהו השלב היקר - **3-6 דקות** - והסיבה לכך ששאר הפרויקט יכול להישאר זול: כל מחברת מאוחרת יותר קוראת את
`edges_traveltime.csv` במקום לקרוא מחדש 816 MB.

המעבר עושה שימוש חוזר בהנחת הסדר שמחברת 02 מצהירה עליה *ומאמתת*: ה-feed ממוין לפי
`(trip_id, stop_sequence)`, ולכן שתי שורות עוקבות של אותו `trip_id` הן שתי תחנות עוקבות של אותו trip.
אנו בודקים זאת שוב כאן ללא עלות נוספת על ידי ספירת נסיגות ב-`stop_sequence` ובלוקים של trip-ים
משולבים זה בזה, בעודנו כבר נוגעים בכל שורה; אם אחד המונים אינו אפס, המשכים שלהלן חסרי משמעות והתא
אומר זאת בקול רם.

עבור כל זוג עוקב `u -> v` בתוך trip אנו מחשבים

```
travel_seconds = arrival_time(v) - departure_time(u)
```

כששניהם מפוענחים לשניות מאז חצות יום השירות. אם `arrival_time` ריק, אנו נסוגים אל `departure_time`
באותה שורה ולהפך, וזה בלתי מזיק ב-feed זה משום ששתי העמודות זהות בכל שורה (מאומת על ידי המונה
`rows_with_dwell` שלהלן) - ה-feed הישראלי אינו ממדל כלל זמן שהייה (dwell time). ראוי לומר זאת
במפורש: **זמני הנסיעה שלנו הם הפרשי לוח זמנים מדלת לדלת ואינם מכילים רכיב שהייה נפרד**, ולכן משך של
קטע הוא "הזמן מיציאה מ-u ועד הגעה ל-v" ותו לא.

ארבעה כללי פסילה, כל אחד נספר בנפרד כדי שסעיף 8 יוכל לדווח בדיוק מה הושלך:

| כלל | מדוע |
|---|---|
| `u == v` | לולאה עצמית, תוצר לוואי של תזמון ב-feed; מחברת 02 מסירה אותן מהגרף אף היא |
| אחד הזמנים חסר/פגום | לא ניתן לחשב משך |
| `duration < MIN_TRAVEL_SECONDS` (כלומר <= 0) | זמן נסיעה אי-חיובי הוא בלתי אפשרי; משמעותו ששתי השורות ממוינות שגוי או חולקות חותמת זמן |
| `duration > MAX_TRAVEL_SECONDS` (3 h) | שום קטע מתוכנן יחיד בין שתי תחנות עוקבות בישראל אינו אורך שלוש שעות; אלו שגיאות feed, בדרך כלל trip שזמניו מתחילים מחדש |

**זיכרון.** המשכים נצברים לכל קטע ב-`array("i")` (4 בתים לתצפית) ולא ברשימות Python (28+ בתים לכל
int). כ-15.3M תצפיות על פני כ-52k קטעים הן אפוא בערך **60 MB של מטען, שיא של כ-150 MB** כולל העותקים
של numpy הנוצרים בסעיף הבא. זהו המקום היחיד שבו המחברת רעבה לזיכרון, והוא חסום על ידי מספר השורות
ב-feed.

In [ ]:
def stream_segment_durations(path, progress_every=PROGRESS_EVERY):
    """One pass over stop_times.txt collecting the travel-time distribution per segment.

    Returns (durations, stats) where durations maps (u, v) -> array('i') of observed
    scheduled travel times in seconds. Memory is O(observations), never O(rows) in
    Python objects.
    """
    durations = defaultdict(lambda: array("i"))
    stats = dict(
        rows_read=0, trips_seen=0, segments_seen=0, kept=0,
        drop_self_loop=0, drop_missing_time=0, drop_nonpositive=0, drop_too_long=0,
        rows_missing_time=0, rows_with_dwell=0, rows_hour_ge_24=0,
        stop_sequence_regressions=0, interleaved_trip_blocks=0,
    )
    seen_trips = set()
    t0 = time.time()

    with open(path, encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        ti = header.index("trip_id")
        si = header.index("stop_id")
        ai = header.index("arrival_time")
        di = header.index("departure_time")
        qi = header.index("stop_sequence") if "stop_sequence" in header else None

        prev_trip = prev_stop = prev_dep = prev_seq = None
        for row in reader:
            stats["rows_read"] += 1
            trip = row[ti]
            stop = row[si]
            arr_raw, dep_raw = row[ai], row[di]
            arr = gtfs_seconds(arr_raw)
            dep = gtfs_seconds(dep_raw)
            if arr is None and dep is None:
                stats["rows_missing_time"] += 1
            elif arr is not None and dep is not None and arr != dep:
                stats["rows_with_dwell"] += 1
            if arr is not None and arr >= 24 * 3600:
                stats["rows_hour_ge_24"] += 1
            # Fall back across the two columns; in this feed they are identical.
            arr_eff = arr if arr is not None else dep
            dep_eff = dep if dep is not None else arr

            if trip != prev_trip:
                if trip in seen_trips:
                    stats["interleaved_trip_blocks"] += 1
                seen_trips.add(trip)
                prev_seq = None
            else:
                stats["segments_seen"] += 1
                if prev_stop == stop:
                    stats["drop_self_loop"] += 1
                elif prev_dep is None or arr_eff is None:
                    stats["drop_missing_time"] += 1
                else:
                    d = arr_eff - prev_dep
                    if d < MIN_TRAVEL_SECONDS:
                        stats["drop_nonpositive"] += 1
                    elif d > MAX_TRAVEL_SECONDS:
                        stats["drop_too_long"] += 1
                    else:
                        durations[(prev_stop, stop)].append(d)
                        stats["kept"] += 1

            if qi is not None:
                try:
                    seq = int(row[qi])
                except (ValueError, IndexError):
                    seq = None
                if seq is not None and prev_seq is not None and seq <= prev_seq:
                    stats["stop_sequence_regressions"] += 1
                prev_seq = seq

            prev_trip, prev_stop, prev_dep = trip, stop, dep_eff

            if progress_every and stats["rows_read"] % progress_every == 0:
                print(f"    {stats['rows_read']:,} rows | {len(durations):,} segments "
                      f"| {time.time() - t0:,.0f}s")

    stats["trips_seen"] = len(seen_trips)
    stats["distinct_segments"] = len(durations)
    stats["elapsed_seconds"] = round(time.time() - t0, 1)
    return durations, stats


print("Streaming stop_times.txt (~15.7M rows; 3-6 minutes) ...")
durations, stream_stats = stream_segment_durations(STOP_TIMES)

print("\nStreaming statistics:")
for k, v in stream_stats.items():
    print(f"  {k:<28} {v:,}" if isinstance(v, int) else f"  {k:<28} {v}")

if stream_stats["stop_sequence_regressions"] or stream_stats["interleaved_trip_blocks"]:
    print("\n" + "!" * 78)
    print("WARNING: the feed is NOT sorted by (trip_id, stop_sequence).")
    print("Consecutive rows are then not consecutive stops, so every duration above is")
    print("meaningless. Sort the file first (see notebook 02) before trusting this stage.")
    print("!" * 78)
else:
    print("\nOrdering assumption holds over the full file - durations are trustworthy.")

## 8. מה נפסל, ומדוע

מפורשוּת לגבי פסילות היא ההבדל בין מערך נתונים מסונן לבין מערך נתונים מוטה בחשאי. ה-`assert` שלהלן
הוא לב הסעיף: הטבלה נותנת דין וחשבון על **כל** זוג תחנות עוקבות שמעבר הזרימה ראה - כל אחד מהם הפך
לתצפית או נפל לדלי פסילה אחד בדיוק, ושני הצדדים חייבים להסתכם במדויק.

המונה שיש להתבונן בו הוא `drop_nonpositive`. הוא נורה כאשר שתי תחנות עוקבות של trip נושאות **אותה
חותמת זמן**, מה שקורה משום שלוח הזמנים מוגדר ברזולוציית דקה בלבד בחלק מהרשת: שתי תחנות המשורתות בתוך
אותה דקה מייצרות הפרש של 0 שניות בדיוק. אלו אינן שורות פגומות, אלא *מגבלת רזולוציה* של מקור הנתונים.
על פרוסה בת 400k שורות מ-feed זה השיעור עמד על כ-0.5% מהזוגות; הטבלה המודפסת נותנת את הנתון האמיתי
לקובץ המלא, וזהו המספר שיש לצטט ולא זה. אופן הטיפול וההטיה הנובעת ממנו:

* אנו פוסלים אותם במקום לקבוע אותם לשנייה אחת, משום שקשת מפוברקת בת שנייה אחת הייתה גורמת לשרשרת של
  תחנות צפופות להיראות כטלפורטציה חינמית עבור כל אלגוריתם מסלול קצר ביותר.
* קטע שורד כל עוד **לפחות trip אחד** עליו הפיק משך חיובי, ולכן פסילת תצפיות באורך אפס מסיטה את ה-median
  של אותו קטע מעט **כלפי מעלה**. קטעים שבהם *כל* תצפית הייתה אפס נעלמים לחלוטין מגרף זמני הנסיעה -
  ואלו יהיו באופן לא-פרופורציונלי הקישורים העירוניים הקצרים והצפופים, כלומר ההטיה אינה אחידה על פני
  הרשת. סעיף 9 מדפיס כמה קטעים משלב 02 אבדו כך.

תרשים העמודות משתמש בציר לוגריתמי משום שדלי הנשמרים גדול בסדרי גודל מן האחרים.

In [ ]:
buckets = {
    "kept (valid observation)": stream_stats["kept"],
    "self-loop (u == v)": stream_stats["drop_self_loop"],
    "missing / malformed time": stream_stats["drop_missing_time"],
    f"non-positive (< {MIN_TRAVEL_SECONDS}s)": stream_stats["drop_nonpositive"],
    f"absurd (> {MAX_TRAVEL_SECONDS // 3600}h)": stream_stats["drop_too_long"],
}
total_pairs = stream_stats["segments_seen"]
assert sum(buckets.values()) == total_pairs, "discard buckets must account for every pair"

discard_df = pd.DataFrame({
    "reason": list(buckets.keys()),
    "observations": list(buckets.values()),
})
discard_df["share_of_pairs"] = (discard_df["observations"] / total_pairs).round(6)
discard_df.to_csv(TABLES / "discard_reasons.csv", index=False, encoding="utf-8-sig")

print(f"consecutive stop pairs seen : {total_pairs:,}")
print(f"kept as observations        : {stream_stats['kept']:,} "
      f"({stream_stats['kept'] / total_pairs:.2%})")
print(f"rows where arrival != departure (dwell modelled): {stream_stats['rows_with_dwell']:,}")
print(f"rows with hour >= 24 (service-day convention)   : {stream_stats['rows_hour_ge_24']:,} "
      f"({stream_stats['rows_hour_ge_24'] / stream_stats['rows_read']:.2%})")
display(discard_df)

fig, ax = plt.subplots(figsize=(9, 4.5))
colors = ["#16a34a"] + ["#dc2626"] * (len(discard_df) - 1)
bars = ax.bar(range(len(discard_df)), discard_df["observations"], color=colors)
ax.set_yscale("log")
ax.set_xticks(range(len(discard_df)))
ax.set_xticklabels(discard_df["reason"], rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Consecutive stop pairs (log scale)")
ax.set_title("Fate of every consecutive stop pair in stop_times.txt")
for b, v in zip(bars, discard_df["observations"]):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{v:,}",
            ha="center", va="bottom", fontsize=8)
ax.margins(y=0.25)
fig.tight_layout()
fig.savefig(FIGURES / "discard_reasons.png", dpi=FIG_DPI)
plt.show()

## 9. מהתפלגויות למשקלי קשתות

כל קטע מחזיק כעת התפלגות מלאה של משכים מתוכננים שנצפו. אנו מסכמים אותה בשלושה מדדי סדר:

* **median** - משקל הקשת. Median ולא ממוצע משום שההתפלגות מוטה ימינה (מקצת ה-trip-ים מתוכננים עם
  המתנות ארוכות) וחריג בודד אינו אמור להזיז משקל.
* **p25 / p75** - התחום הבין-רבעוני, כלומר עד כמה אותה קשת *אמינה*. קטע שה-p25 וה-p75 שלו הם 4 ו-6
  דקות הוא קישור עקבי; קטע עם 2 ו-25 דקות משמעו שלוח הזמנים משתנה עצומות לפי שעת היום או לפי הקו, וכל
  משקל חד-מספרי עבורו הוא פישוט.
  ייצוא הרבעונים מאפשר למחברות 20-22 לכמת אי-ודאות זו במקום להעמיד פנים שה-median מדויק.

אנו רושמים גם את `n_observations` (כמה trip-ים תקפים הפיקו את האומדן) לצד `trip_frequency` ממחברת 02
(כמה trip-ים חוצים את הקטע בסך הכל). הפער בין שתי העמודות *הוא* שיעור הפסילה עבור אותה קשת ספציפית,
כך שקורא יכול לזהות קשת שה-median שלה נשען על שלוש תצפיות ששרדו מתוך ארבע מאות trip-ים.

משקל הגרף **הלא-מכוון** מחושב על ידי איגום התצפיות הגולמיות של שני כיווני הנסיעה ולקיחת ה-median של
הקבוצה המאוגמת - ולא על ידי מיצוע שני ה-median-ים הכיווניים. האיגום מדויק ומשקלל אוטומטית כל כיוון
לפי תדירות השירות בפועל.

`edges_traveltime.csv` נכתב מכוון (שורה אחת לכל `from_stop -> to_stop`), בהתאמה לאוריינטציה של
`edges.csv` ממחברת 02; הגרף המשומר (pickled) הוא לא-מכוון, בהתאמה ל-`graph_undirected.pkl`. שתי
המוסכמות הן מה שהמחברות שבהמשך הזרם מצפות לו.

In [ ]:
records = []
pooled = defaultdict(list)     # (min_id, max_id) -> list of numpy arrays

for (u, v), arr in durations.items():
    if len(arr) < MIN_OBSERVATIONS:
        continue
    a = np.asarray(arr, dtype=np.int32)
    p25, med, p75 = np.percentile(a, [25, 50, 75])
    records.append({
        "from_stop": u,
        "to_stop": v,
        "median_travel_seconds": float(round(med, 1)),
        "p25_travel_seconds": float(round(p25, 1)),
        "p75_travel_seconds": float(round(p75, 1)),
        "trip_frequency": int(FREQ.get((u, v), len(arr))),
        "n_observations": int(len(arr)),
    })
    pooled[(u, v) if u <= v else (v, u)].append(a)

tt_edges = pd.DataFrame.from_records(records)
tt_edges = tt_edges[["from_stop", "to_stop", "median_travel_seconds",
                     "p25_travel_seconds", "p75_travel_seconds",
                     "trip_frequency", "n_observations"]]
tt_edges.to_csv(TABLES / "edges_traveltime.csv", index=False, encoding="utf-8-sig")

# How many stage-02 segments could not be given a travel time at all?
missing_directed = sorted(set(FREQ) - set(zip(tt_edges["from_stop"], tt_edges["to_stop"])))
print(f"directed segments with a travel time : {len(tt_edges):,}")
print(f"stage-02 segments with NO valid observation: {len(missing_directed):,} "
      f"({len(missing_directed) / max(len(FREQ), 1):.2%} of stage-02 edges)")
print(f"total observations behind the table  : {tt_edges['n_observations'].sum():,}")
print(f"median observations per segment      : {tt_edges['n_observations'].median():.0f}")
print(f"segments resting on a single observation: "
      f"{(tt_edges['n_observations'] == 1).sum():,}")
display(tt_edges.head(TOP_N))

### בניית גרף זמני הנסיעה ושמירתו

ההיטל הלא-מכוון מאגם את שני הכיוונים, מצרף את תכונות התחנות ממחברת 02, ושומר שלוש תכונות קשת:

* `travel_seconds` - ה-median המאוגם, הגודל בעל המשמעות;
* `weight` - **שיקוף של `travel_seconds`**, כך שכל קריאה ל-`networkx` שברירת המחדל שלה היא
  `weight="weight"` תבצע ניתוב מבוסס זמן ולא ניתוב מבוסס תדירות. זוהי שבירה מכוונת של המוסכמה ב-
  `graph_undirected.pkl`, שבו `weight` הוא תדירות ה-trip-ים. מחברות שבהמשך הזרם הטוענות את
  `graph_traveltime.pkl` מקבלות גרף *זמן* וחייבות להתייחס ל-`weight` כשניות;
* `trip_frequency` - תדירות שלב 02, מסוכמת על פני שני הכיוונים, נשמרת כדי שמחברת הזקוקה לשני הגדלים
  לא תצטרך לבצע join מחדש עם ה-CSV.

`del durations` משחרר את כ-60 MB של המערכים שנצברו לפני סעיפי ה-betweenness, שהם צרכן הזיכרון הבא.

In [ ]:
Gt = nx.Graph()
for (a, b), arrs in pooled.items():
    all_obs = np.concatenate(arrs)
    med = float(np.median(all_obs))
    freq = int(FREQ.get((a, b), 0)) + int(FREQ.get((b, a), 0))
    Gt.add_edge(a, b,
                travel_seconds=med,
                weight=med,                 # mirror: nx defaults to weight="weight"
                trip_frequency=freq,
                n_observations=int(all_obs.size))

for n in Gt.nodes():
    Gt.nodes[n].update(ATTR.get(n, DEFAULT_ATTR))

del durations, pooled

with open(STAGE / "graph_traveltime.pkl", "wb") as f:
    pickle.dump(Gt, f)

components = sorted(nx.connected_components(Gt), key=len, reverse=True)
Gc = Gt.subgraph(components[0]).copy()
lcc_share = Gc.number_of_nodes() / Gt.number_of_nodes()

print(f"travel-time graph : {Gt.number_of_nodes():,} nodes, {Gt.number_of_edges():,} edges")
print(f"connected components: {len(components):,}; "
      f"largest holds {Gc.number_of_nodes():,} nodes ({lcc_share:.2%})")
print(f"saved: {STAGE / 'graph_traveltime.pkl'} "
      f"({(STAGE / 'graph_traveltime.pkl').stat().st_size / 1024**2:.1f} MB)")

# Explicit comparison against the hop graph of notebook 02.
hop_nodes = set(nodes_df["stop_id"].astype(str))
print(f"\nstage-02 hop graph nodes: {len(hop_nodes):,}")
print(f"nodes lost to the travel-time filter: {len(hop_nodes - set(Gt.nodes())):,} "
      "(all of their segments had zero or unusable durations)")

## 10. כיצד נראים זמני הנסיעה?

שלוש תצוגות שפיות לפני שאנו נותנים אמון במשקלים:

1. **התפלגות זמן הנסיעה ה-median לכל קטע** (ציר y לוגריתמי). רשת תחבורה ציבורית אמורה להיות נשלטת על
   ידי קפיצות עירוניות קצרות של דקה עד שלוש עם זנב דק של קטעים בין-עירוניים. כל תוצאה אחרת - מוד באפס,
   שיא דו-מודלי בתקרת 3 השעות - הייתה מעידה שהפענוח או המסננים שגויים.
2. **הפיזור הבין-רבעוני היחסי**, `(p75 - p25) / median`, המבטא כמה משתנה משכו המתוכנן של קטע בין
   trip-ים. ערכים קרובים לאפס משמעם שלוח הזמנים מתייחס לקטע כקבוע; ערכים גדולים משמעם ש-median יחיד
   הוא סיכום ירוד של אותה קשת.
3. **זמן נסיעה מול תדירות שירות**, על צירים לוג-לוג: האם הקטעים המשורתים בכבדות נוטים להיות הקצרים
   והעירוניים? זוהי הבדיקה ששני השקלולים אכן מודדים דברים שונים - אילו זמן הנסיעה היה רק פונקציה
   מונוטונית של התדירות, כל תרגיל השקלול מחדש היה מיותר.

In [ ]:
med_s = tt_edges["median_travel_seconds"].to_numpy(dtype=float)
rel_iqr = ((tt_edges["p75_travel_seconds"] - tt_edges["p25_travel_seconds"])
           / tt_edges["median_travel_seconds"].replace(0, np.nan)).to_numpy(dtype=float)

pct = np.percentile(med_s, [1, 25, 50, 75, 90, 99])
print("Median travel time per segment (seconds): "
      f"p1={pct[0]:.0f}  p25={pct[1]:.0f}  p50={pct[2]:.0f}  "
      f"p75={pct[3]:.0f}  p90={pct[4]:.0f}  p99={pct[5]:.0f}")
print(f"in minutes: p50={pct[2] / 60:.1f}  p90={pct[4] / 60:.1f}  p99={pct[5] / 60:.1f}")
print(f"segments over 30 minutes: {(med_s > 1800).sum():,} "
      f"({(med_s > 1800).mean():.2%})")
print(f"median relative IQR (p75-p25)/median: {np.nanmedian(rel_iqr):.2f}")
print(f"segments with a zero-width IQR (identical on every trip): "
      f"{(np.nan_to_num(rel_iqr) == 0).sum():,}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

axes[0].hist(med_s / 60.0, bins=80, range=(0, 60), color="#2563eb",
             edgecolor="white", linewidth=0.3)
axes[0].set_yscale("log")
axes[0].set_xlabel("Median travel time (minutes, truncated at 60)")
axes[0].set_ylabel("Number of segments (log scale)")
axes[0].set_title("Segment travel-time distribution")

axes[1].hist(rel_iqr[np.isfinite(rel_iqr)], bins=60, range=(0, 3), color="#d97706",
             edgecolor="white", linewidth=0.3)
axes[1].set_yscale("log")
axes[1].set_xlabel("(p75 - p25) / median")
axes[1].set_ylabel("Number of segments (log scale)")
axes[1].set_title("How stable is each segment's schedule?")

sample = tt_edges.sample(min(20000, len(tt_edges)), random_state=PATH_SEED)
axes[2].scatter(sample["trip_frequency"].clip(lower=1),
                sample["median_travel_seconds"].clip(lower=1),
                s=3, alpha=0.15, color="#7c3aed")
axes[2].set_xscale("log")
axes[2].set_yscale("log")
axes[2].set_xlabel("Trip frequency (trips/day, both endpoints as ordered)")
axes[2].set_ylabel("Median travel time (s)")
axes[2].set_title("Frequency vs travel time")

fig.tight_layout()
fig.savefig(FIGURES / "traveltime_distribution.png", dpi=FIG_DPI)
plt.show()

rho_freq_time = (tt_edges["trip_frequency"].corr(tt_edges["median_travel_seconds"],
                                                 method="spearman"))
print(f"\nSpearman(trip_frequency, median_travel_seconds) = {rho_freq_time:.3f} "
      "- if this were near +/-1 the re-weighting would add nothing.")

## 11. האם מסלולים קצרים ביותר לפי זמן נסיעה שונים ממסלולים קצרים ביותר לפי ספירת קפיצות?

זהו הראשון מבין שני המבחנים שקובעים אם המחברות הקודמות זקוקות להסתייגות.

אנו מגרילים `N_PATH_PAIRS` זוגות מוצא-יעד אקראיים מן הרכיב הגדול ביותר, ולכל אחד מהם מחשבים שני
מסלולים **על אותו גרף**:

* המסלול הקצר ביותר לפי **ספירת קפיצות** (BFS לא-משוקלל) - מה שכל מחברת עד כה הניחה במובלע שהנוסע
  עושה;
* המסלול הקצר ביותר לפי **זמן נסיעה** (Dijkstra על `travel_seconds`) - מה שנוסע הממזער את זמן המסע
  היה עושה בפועל.

לאחר מכן אנו מנקדים אותם זה מול זה:

* `same_path` - האם שני המסלולים הם פשוטו כמשמעו אותה סדרת תחנות?
* `hop_path_seconds` מול `time_path_seconds` - **כמה זמן מפסיד נוסע העוקב אחר המסלול האופטימלי
  בקפיצות.** זהו המספר המכמת את עלות בחירת המידול הקודמת.
* `time_path_hops` מול `hop_path_hops` - דרך כמה תחנות נוספות מוכן המסלול האופטימלי בזמן לעבור כדי
  לחסוך את אותן דקות.

עלות: כ-1-3 דקות עבור 300 זוגות. שתי השאילתות הן חיפושים דו-כיווניים, ולכן כל אחת זולה בהרבה מהרצה
מלאה ממקור יחיד, אך Dijkstra ב-`networkx` בפייתון טהור עודנו החצי האיטי.

In [ ]:
rng = random.Random(PATH_SEED)
lcc_nodes = list(Gc.nodes())

pairs = []
while len(pairs) < N_PATH_PAIRS:
    s, t = rng.choice(lcc_nodes), rng.choice(lcc_nodes)
    if s != t:
        pairs.append((s, t))


def path_seconds(graph, path):
    return sum(graph[a][b]["travel_seconds"] for a, b in zip(path, path[1:]))


t0 = time.time()
rows = []
for s, t in pairs:
    hop_path = nx.shortest_path(Gc, s, t)                              # BFS
    time_path = nx.shortest_path(Gc, s, t, weight="travel_seconds")    # Dijkstra
    rows.append({
        "source": s, "target": t,
        "hop_path_hops": len(hop_path) - 1,
        "time_path_hops": len(time_path) - 1,
        "hop_path_seconds": round(path_seconds(Gc, hop_path), 1),
        "time_path_seconds": round(path_seconds(Gc, time_path), 1),
        "same_path": hop_path == time_path,
    })
print(f"{len(rows)} origin-destination pairs routed twice in {time.time() - t0:.1f}s")

paths = pd.DataFrame(rows)
paths["extra_seconds_if_hop_routed"] = paths["hop_path_seconds"] - paths["time_path_seconds"]
paths["extra_hops_if_time_routed"] = paths["time_path_hops"] - paths["hop_path_hops"]
paths["time_penalty_ratio"] = (paths["hop_path_seconds"]
                               / paths["time_path_seconds"].replace(0, np.nan))
paths.to_csv(TABLES / "path_comparison.csv", index=False, encoding="utf-8-sig")

same_share = paths["same_path"].mean()
med_penalty = paths["extra_seconds_if_hop_routed"].median()
mean_penalty = paths["extra_seconds_if_hop_routed"].mean()
med_ratio = paths["time_penalty_ratio"].median()
p90_penalty = paths["extra_seconds_if_hop_routed"].quantile(0.90)

print(f"\nidentical routes                        : {same_share:.1%} of pairs")
print(f"median time lost by hop-routing         : {med_penalty / 60:.1f} min")
print(f"mean time lost by hop-routing           : {mean_penalty / 60:.1f} min")
print(f"90th percentile time lost               : {p90_penalty / 60:.1f} min")
print(f"median hop-path / time-path duration    : {med_ratio:.2f}x")
print(f"median extra stops on the time-optimal route: "
      f"{paths['extra_hops_if_time_routed'].median():.0f}")
print(f"pairs where the time-optimal route is LONGER in hops: "
      f"{(paths['extra_hops_if_time_routed'] > 0).mean():.1%}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
lim = max(paths["hop_path_seconds"].max(), paths["time_path_seconds"].max()) / 60 * 1.05
axes[0].scatter(paths["time_path_seconds"] / 60, paths["hop_path_seconds"] / 60,
                s=14, alpha=0.5, color="#2563eb")
axes[0].plot([0, lim], [0, lim], ls="--", lw=1, color="#334155",
             label="identical duration")
axes[0].set_xlim(0, lim); axes[0].set_ylim(0, lim)
axes[0].set_xlabel("Time-optimal route (minutes)")
axes[0].set_ylabel("Hop-optimal route (minutes)")
axes[0].set_title("Cost of routing by hop count\n(points above the line = time wasted)")
axes[0].legend()

axes[1].hist(paths["extra_hops_if_time_routed"], bins=range(
    int(paths["extra_hops_if_time_routed"].min()) - 1,
    int(paths["extra_hops_if_time_routed"].max()) + 2),
    color="#16a34a", edgecolor="white", linewidth=0.4)
axes[1].set_xlabel("Extra stops on the time-optimal route")
axes[1].set_ylabel("Number of origin-destination pairs")
axes[1].set_title("How many more stops a time-minimising passenger accepts")

fig.tight_layout()
fig.savefig(FIGURES / "path_comparison.png", dpi=FIG_DPI)
plt.show()

## 12. Betweenness על גרף זמני הנסיעה - ובקרה כנה

Betweenness הוא המדד שבו הפרויקט משתמש כדי לכנות תחנה "קריטית", ולכן זהו המדד המשמעותי כאן. אנו
מחשבים אותו מחדש עם `travel_seconds` כמשקל הקשת, מה שגורם ל-`networkx` להריץ ספירת מסלולים קצרים
ביותר משוקללת (Dijkstra) במקום BFS.

**הבקרה חשובה יותר מן הכותרת.** השוואת האומדן החדש שלנו ישירות מול `approx_betweenness` ממחברת 04
הייתה מערבבת שני דברים שונים: שינוי השקלול, והעובדה ששתי ההרצות דגמו צמתי מקור *שונים* (מחברת 04
השתמשה ב-`k=300`; betweenness מדגמי הוא רועש, ובדיקת יציבות ה-seed של מחברת 04 עצמה כימתה את הרעש
הזה). לפיכך אנו מריצים **שלושה** מספרים:

| עמודה | מה זה |
|---|---|
| `bt_traveltime` | betweenness משוקלל, `k=K_BETWEENNESS`, seed `BETWEENNESS_SEED` |
| `bt_hop_control` | betweenness **לא-משוקלל** על *אותו גרף*, *אותו k*, *אותו seed* - ולכן אותה קבוצת מקורות שנדגמה. כל הבדל מ-`bt_traveltime` נגרם אך ורק מן השקלול |
| `bt_hop_nb04` | ה-`approx_betweenness` שפורסם במחברת 04 (k=300, מדגם שונה, ומחושב על גרף הקפיצות משלב 02, שיש בו מעט יותר קשתות) |

ההשוואה המדעית הנקייה היא `bt_traveltime` מול `bt_hop_control`. `bt_hop_nb04` מדווח משום שזה מה
שהמחברות הקודמות והדוח הכתוב השתמשו בו בפועל, ולכן ההסכמה שלו - או היעדרה - היא המספר הרלוונטי
מעשית.

**עלות: 5-15 דקות להרצה המשוקללת בתוספת 1-3 דקות לבקרה.** קבעו את `RUN_BETWEENNESS` ל-`False` כדי
לדלג על סעיפים 12-14; פלטי החוזה של מחברת זו
(`edges_traveltime.csv`, `graph_traveltime.pkl`) כבר נכתבו בשלב זה.

In [ ]:
bt_tt = bt_hop = None
k_eff = None

if RUN_BETWEENNESS:
    k_eff = min(K_BETWEENNESS, Gc.number_of_nodes())
    print(f"LCC: {Gc.number_of_nodes():,} nodes / {Gc.number_of_edges():,} edges | "
          f"k = {k_eff} sources ({k_eff / Gc.number_of_nodes():.2%} of the LCC)")

    t0 = time.time()
    bt_tt = nx.betweenness_centrality(Gc, k=k_eff, seed=BETWEENNESS_SEED,
                                      normalized=True, weight="travel_seconds")
    print(f"  travel-time (weighted) betweenness : {time.time() - t0:,.0f}s")

    t0 = time.time()
    bt_hop = nx.betweenness_centrality(Gc, k=k_eff, seed=BETWEENNESS_SEED,
                                       normalized=True, weight=None)
    print(f"  hop-count control (same k, same seed, same sources): {time.time() - t0:,.0f}s")

    nz_tt = sum(1 for v in bt_tt.values() if v > 0)
    nz_hop = sum(1 for v in bt_hop.values() if v > 0)
    print(f"\nnon-zero estimates: travel-time {nz_tt:,} | hop control {nz_hop:,} "
          f"of {len(bt_tt):,} stations")
    print("Stations at zero were simply never on a sampled shortest path - with k = "
          f"{k_eff} sources that is expected, not a finding.")
else:
    print("RUN_BETWEENNESS is False - sections 12-14 are skipped. "
          "The contract outputs of this notebook are already written.")

## 13. האם הדירוג שורד את השקלול מחדש?

כעת השאלה עצמה. אנו מבצעים join של שלוש עמודות ה-betweenness על `stop_id` ומדווחים, עבור כל זוג:

* **Spearman rho על פני כל תחנות ה-LCC** - מתאם הדירוגים הכולל את המסה הגדולה של תחנות שקיבלו ציון אפס
  תחת שני האומדנים. שוויונות (ties) באפס מנפחים מספר זה, ולכן זוהי הקריאה האופטימית.
* **Spearman rho על פני תחנות החיוביות תחת אחד האומדנים לפחות** - הקריאה הכנה, משום שהיא מודדת הסכמה
  היכן שיש בכלל על מה לחלוק.
* **חפיפת top-50 ו-top-10** - המספר הרלוונטי מעשית. טענות הפרויקט עוסקות ברשימה מצומצמת של תחנות
  קריטיות, ולכן מה שחשוב הוא אם הרשימה המצומצמת הזו היא אותה רשימה.

מדריך פרשנות, המוצהר מראש כדי שלא ניתן יהיה להתאימו בדיעבד: חפיפת top-50 מעל כ-0.8 משמעה שהרשימה
המצומצמת הקודמת מבוססת הקפיצות בטוחה באופן כללי; בין 0.5 ל-0.8 משמעה שהיא נכונה בכיוונה אך טענות על
תחנות בודדות שבירות; מתחת ל-0.5 משמעה שדירוג הקפיצות ודירוג זמני הנסיעה הם אובייקטים שונים מהותית,
וכל אמירה קודמת מסוג "התחנה הקריטית ביותר" זקוקה לצירוף ההסתייגות.

In [ ]:
if RUN_BETWEENNESS:
    nb04 = pd.read_csv(METRICS_CSV, dtype={"stop_id": str}, encoding="utf-8-sig")
    if "approx_betweenness" not in nb04.columns:
        raise KeyError("stop_metrics.csv has no 'approx_betweenness' column - "
                       "re-run notebook 04_centrality_analysis.")
    nb04_bt = dict(zip(nb04["stop_id"], nb04["approx_betweenness"].astype(float)))

    bt = pd.DataFrame({
        "stop_id": list(Gc.nodes()),
    })
    bt["stop_name"] = bt["stop_id"].map(lambda n: Gc.nodes[n].get("stop_name", ""))
    bt["region"] = bt["stop_id"].map(lambda n: Gc.nodes[n].get("region", ""))
    bt["lat"] = bt["stop_id"].map(lambda n: Gc.nodes[n].get("lat", np.nan))
    bt["lon"] = bt["stop_id"].map(lambda n: Gc.nodes[n].get("lon", np.nan))
    bt["bt_traveltime"] = bt["stop_id"].map(bt_tt).fillna(0.0)
    bt["bt_hop_control"] = bt["stop_id"].map(bt_hop).fillna(0.0)
    bt["bt_hop_nb04"] = bt["stop_id"].map(nb04_bt).fillna(0.0)
    bt["rank_traveltime"] = bt["bt_traveltime"].rank(ascending=False, method="min").astype(int)
    bt["rank_hop_control"] = bt["bt_hop_control"].rank(ascending=False, method="min").astype(int)
    bt["rank_hop_nb04"] = bt["bt_hop_nb04"].rank(ascending=False, method="min").astype(int)
    bt["rank_shift_vs_control"] = bt["rank_hop_control"] - bt["rank_traveltime"]
    bt["rank_shift_vs_nb04"] = bt["rank_hop_nb04"] - bt["rank_traveltime"]
    bt = bt.sort_values("bt_traveltime", ascending=False).reset_index(drop=True)
    bt.to_csv(TABLES / "betweenness_traveltime.csv", index=False, encoding="utf-8-sig")

    missing_in_nb04 = int((~bt["stop_id"].isin(set(nb04_bt))).sum())

    def agreement(col_a, col_b, label):
        rho_all = bt[col_a].corr(bt[col_b], method="spearman")
        sub = bt[(bt[col_a] > 0) | (bt[col_b] > 0)]
        rho_pos = sub[col_a].corr(sub[col_b], method="spearman") if len(sub) > 2 else np.nan
        out = {"comparison": label, "n_stations": len(bt),
               "n_positive_either": len(sub),
               "spearman_all": round(float(rho_all), 4),
               "spearman_positive_either": round(float(rho_pos), 4)}
        for n in (10, 50, 100):
            a = set(bt.nlargest(n, col_a)["stop_id"])
            b = set(bt.nlargest(n, col_b)["stop_id"])
            out[f"top{n}_overlap"] = round(len(a & b) / n, 3)
        return out

    agree = pd.DataFrame([
        agreement("bt_traveltime", "bt_hop_control",
                  "travel-time vs hop control (same sample - clean)"),
        agreement("bt_traveltime", "bt_hop_nb04",
                  "travel-time vs notebook 04 (different sample - practical)"),
        agreement("bt_hop_control", "bt_hop_nb04",
                  "hop control vs notebook 04 (sampling noise floor)"),
    ])
    agree.to_csv(TABLES / "betweenness_agreement.csv", index=False, encoding="utf-8-sig")
    print(f"stations in the travel-time LCC missing from notebook 04: {missing_in_nb04:,}")
    display(agree.T)
else:
    bt = agree = None

### קריאת השורה השלישית לפני שתי הראשונות

השורה התחתונה באותה טבלה - **בקרת הקפיצות מול מחברת 04** - היא *רצפת הרעש*. שתיהן betweenness מבוסס
קפיצות; הן נבדלות זו מזו רק משום שדגמו מקורות שונים (ומשום שמחברת 04 רצה על גרף שלב 02 המעט גדול
יותר). כל אי-הסכמה שאותה שורה מראה היא כמות אי-ההסכמה שהדגימה לבדה מייצרת.

השקלול מחדש מלמד אותנו משהו רק אם שורות זמן הנסיעה חולקות **יותר** מאשר רצפת הרעש. אם כל שלוש השורות
נראות דומות, המסקנה הנכונה היא "איננו יכולים להפריד את השפעת השקלול מרעש האומדן ב-k = 200", והתיקון
יהיה `K_BETWEENNESS` גדול יותר, ולא טענה חזקה יותר. תרשימי הפיזור שלהלן מציגים את אותן שלוש השוואות
על דירוגים, שם הרעש קל יותר לזיהוי מאשר על הערכים הגולמיים.

In [ ]:
if RUN_BETWEENNESS:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5.2))
    comparisons = [
        ("rank_hop_control", "rank_traveltime", "Hop control rank", "Travel-time rank",
         "Same sample, only the weighting differs", "#2563eb"),
        ("rank_hop_nb04", "rank_traveltime", "Notebook 04 rank", "Travel-time rank",
         "What the earlier notebooks used", "#dc2626"),
        ("rank_hop_nb04", "rank_hop_control", "Notebook 04 rank", "Hop control rank",
         "Sampling noise floor (both hop-count)", "#94a3b8"),
    ]
    pool = bt.nsmallest(RANK_POOL, "rank_traveltime")
    for ax, (xc, yc, xl, yl, title, color) in zip(axes, comparisons):
        ax.scatter(bt[xc], bt[yc], s=4, alpha=0.15, color=color)
        ax.scatter(pool[xc], pool[yc], s=12, alpha=0.7, color="#111827",
                   label=f"top {RANK_POOL} by travel time")
        lim = len(bt)
        ax.plot([1, lim], [1, lim], ls="--", lw=1, color="#334155")
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlabel(xl); ax.set_ylabel(yl)
        ax.set_title(title, fontsize=11)
        ax.legend(fontsize=8, loc="lower right")
    fig.suptitle("Betweenness rank agreement (log-log; closer to the diagonal = more agreement)")
    fig.tight_layout()
    fig.savefig(FIGURES / "betweenness_rank_agreement.png", dpi=FIG_DPI)
    plt.show()

## 14. הזזות הדירוג הגדולות ביותר

מתאמים מצרפיים מסתירים את המקרים המעניינים, ולכן אנו נוקבים בשמם. ניתוח ההזזות מוגבל למאגר התחנות
המדורגות ב-top `RANK_POOL` תחת **אחד** השקלולים לפחות: מחוץ למאגר זה כמעט לכל תחנה יש betweenness של
אפס בדיוק תחת מדגם `k = 200`, וה"דירוג" שלה הוא שבירת שוויון שרירותית ולא מדידה. דירוג הזזות על פני
כל 30k התחנות היה מייצר טבלה של רעש טהור.

`rank_shift_vs_control` הוא `rank_hop_control - rank_traveltime`, ולכן:

* **חיובי** = התחנה *מרכזית יותר* ברגע שדקות מחליפות קפיצות. אלו אמורות להיות צמתי מעבר על מסדרונות
  מהירים - תחנות רכבת ומסופים בין-עירוניים, שבהם קשת אחת מכסה מרחק רב במהירות, ולכן מסלולים ממזערי-זמן
  מנותבים דרכן.
* **שלילי** = התחנה הייתה תוצר לוואי של ספירת קפיצות. אלו אמורות להיות תחנות רגילות על שרשראות ארוכות
  של תחנות עירוניות צפופות, הנראות כקיצורי דרך יעילים כשכל קשת עולה 1 אך הן איטיות במציאות.

אם ההזזות **אינן** נופלות לשתי משפחות אלה, השקלול מחדש מייצר רעש ולא אות, ויש לקרוא את הטבלה בהתאם.

In [ ]:
if RUN_BETWEENNESS:
    in_pool = (bt["rank_traveltime"] <= RANK_POOL) | (bt["rank_hop_control"] <= RANK_POOL)
    movers = bt.loc[in_pool].copy()
    movers["abs_shift"] = movers["rank_shift_vs_control"].abs()
    movers = movers.sort_values("abs_shift", ascending=False)

    cols = ["stop_id", "stop_name", "region", "bt_traveltime", "bt_hop_control",
            "rank_traveltime", "rank_hop_control", "rank_shift_vs_control",
            "rank_shift_vs_nb04"]
    movers[cols + ["abs_shift"]].to_csv(
        TABLES / "betweenness_rank_movers.csv", index=False, encoding="utf-8-sig")

    print(f"mover pool: {len(movers):,} stations (top {RANK_POOL} under either weighting)")
    print(f"median |rank shift| inside the pool: "
          f"{movers['abs_shift'].median():.0f} places")
    print("\nRose most under travel-time weighting (gained importance):")
    display(movers.nlargest(TOP_N, "rank_shift_vs_control")[cols])
    print("Fell most under travel-time weighting (were hop-count artefacts):")
    display(movers.nsmallest(TOP_N, "rank_shift_vs_control")[cols])

    top_movers = movers.nlargest(2 * TOP_N, "abs_shift").sort_values("rank_shift_vs_control")
    labels = [f"{(n or sid)} ({sid})" for n, sid
              in zip(top_movers["stop_name"], top_movers["stop_id"])]
    colors = ["#dc2626" if v < 0 else "#16a34a"
              for v in top_movers["rank_shift_vs_control"]]

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.barh(range(len(top_movers)), top_movers["rank_shift_vs_control"], color=colors)
    ax.set_yticks(range(len(top_movers)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.axvline(0, color="#111827", lw=1)
    ax.set_xlabel("Rank shift (hop-count rank - travel-time rank);  "
                  "positive = more central under travel time")
    ax.set_title(f"Biggest betweenness rank movers (top {RANK_POOL} pool, same source sample)")
    fig.tight_layout()
    fig.savefig(FIGURES / "betweenness_rank_movers.png", dpi=FIG_DPI)
    plt.show()

### היכן ממוקמות שתי הרשימות המצומצמות גאוגרפית

בדיקה ויזואלית אחרונה: הרשימה המצומצמת של 50 התחנות המובילות תחת כל שקלול, משורטטת על מפת הארץ. אם
הרשימה המצומצמת של זמני הנסיעה נמשכת אל מסדרונות רכבת ומסופים בין-עירוניים בעוד הרשימה המצומצמת של
ספירת הקפיצות יושבת בתוך שרשראות תחנות עירוניות צפופות, זוהי החתימה הגאוגרפית של האפקט שתואר לעיל,
והיא הופכת את התוצאה המספרית לניתנת לפרשנות ולא רק למתאם.

In [ ]:
if RUN_BETWEENNESS:
    geo = bt.dropna(subset=["lat", "lon"])
    top_tt = geo.nsmallest(50, "rank_traveltime")
    top_hop = geo.nsmallest(50, "rank_hop_control")
    shared = set(top_tt["stop_id"]) & set(top_hop["stop_id"])

    fig, ax = plt.subplots(figsize=(8, 11))
    ax.scatter(geo["lon"], geo["lat"], s=2, alpha=0.12, color="#94a3b8",
               label="all stations")
    ax.scatter(top_hop["lon"], top_hop["lat"], s=55, facecolors="none",
               edgecolors="#dc2626", linewidths=1.4, label="top 50 - hop count")
    ax.scatter(top_tt["lon"], top_tt["lat"], s=18, color="#2563eb",
               label="top 50 - travel time", zorder=5)
    ax.set_aspect(1 / np.cos(np.deg2rad(float(geo["lat"].mean()))))
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    ax.set_title("Top-50 critical stations under each weighting\n"
                 f"({len(shared)} of 50 appear in both lists)")
    ax.legend(loc="lower right", fontsize=9)
    fig.tight_layout()
    fig.savefig(FIGURES / "top50_shortlist_map.png", dpi=FIG_DPI)
    plt.show()
    print(f"stations on both shortlists: {len(shared)} / 50")

## 15. שמירת סיכום השלב

`traveltime_summary.json` הוא הרישום קריא-המכונה של שלב זה: גודל הגרף, כל מוני הפסילות, אחוזוני
התפלגות זמני הנסיעה, תוצאת השוואת המסלולים וטבלת הסכמת ה-betweenness המלאה. מחברות 20-22 והדוח הכתוב
קוראים אותו במקום לגזור מחדש את המספרים, והוא מהווה את שובל הביקורת (audit trail) עבור החלטות הסינון
שהתקבלו בסעיף 7.

In [ ]:
summary = {
    "stage": "18_travel_time_network",
    "edge_weight": "median scheduled travel time in seconds (attribute 'travel_seconds', "
                   "mirrored to 'weight')",
    "thresholds": {
        "min_travel_seconds": MIN_TRAVEL_SECONDS,
        "max_travel_seconds": MAX_TRAVEL_SECONDS,
        "min_observations": MIN_OBSERVATIONS,
    },
    "streaming": stream_stats,
    "discards": {k: int(v) for k, v in buckets.items()},
    "graph": {
        "nodes": Gt.number_of_nodes(),
        "undirected_edges": Gt.number_of_edges(),
        "directed_edges_with_traveltime": int(len(tt_edges)),
        "stage02_segments_without_traveltime": int(len(missing_directed)),
        "connected_components": len(components),
        "largest_component_nodes": Gc.number_of_nodes(),
        "largest_component_share": round(lcc_share, 4),
    },
    "travel_time_seconds": {
        "p1": float(pct[0]), "p25": float(pct[1]), "p50": float(pct[2]),
        "p75": float(pct[3]), "p90": float(pct[4]), "p99": float(pct[5]),
        "median_relative_iqr": float(round(np.nanmedian(rel_iqr), 4)),
        "spearman_frequency_vs_time": float(round(rho_freq_time, 4)),
    },
    "path_comparison": {
        "pairs": int(len(paths)),
        "identical_route_share": float(round(same_share, 4)),
        "median_seconds_lost_by_hop_routing": float(med_penalty),
        "mean_seconds_lost_by_hop_routing": float(round(mean_penalty, 1)),
        "p90_seconds_lost_by_hop_routing": float(p90_penalty),
        "median_duration_ratio": float(round(med_ratio, 4)),
        "median_extra_hops_on_time_route":
            float(paths["extra_hops_if_time_routed"].median()),
    },
    "betweenness": (
        {
            "k_samples": int(k_eff),
            "seed": BETWEENNESS_SEED,
            "agreement": agree.to_dict("records"),
            "median_abs_rank_shift_in_pool": float(movers["abs_shift"].median()),
            "rank_pool": RANK_POOL,
        }
        if RUN_BETWEENNESS else {"skipped": True}
    ),
}

with open(STAGE / "traveltime_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

written = [
    TABLES / "edges_traveltime.csv",
    STAGE / "graph_traveltime.pkl",
    STAGE / "traveltime_summary.json",
    TABLES / "discard_reasons.csv",
    TABLES / "path_comparison.csv",
]
if RUN_BETWEENNESS:
    written += [TABLES / "betweenness_traveltime.csv",
                TABLES / "betweenness_agreement.csv",
                TABLES / "betweenness_rank_movers.csv"]

print("Written:")
for p in written:
    print(f"  {p}  ({p.stat().st_size / 1024:,.0f} KB)")
print("\nSummary (graph + path comparison):")
print(json.dumps({k: summary[k] for k in ("graph", "path_comparison")},
                 indent=2, ensure_ascii=False))

## מסקנות

* **לפרויקט יש כעת גרף משוקלל בזמן, ומחברות 20-22 תלויות בו.**
  `edges_traveltime.csv` מעניק לכל קטע משך מתוכנן median, p25 ו-p75 יחד עם מספר התצפיות שמאחוריו,
  ו-`graph_traveltime.pkl` נושא את ה-median כ-`travel_seconds` (משוקף ל-`weight`). כל אמירה שבהמשך
  הזרם מן הסוג "הסרת תחנה זו מוסיפה N דקות למסע הממוצע" ניתנת לחישוב רק על גרף זה; על גרף הקפיצות היא
  לא הייתה אפילו ניתנת להגדרה.

* **קראו את מוני הפסילות לפני שאתם נותנים אמון בקשת בודדת.** הפסילות כאן אינן שחיתות נתונים, הן
  *רזולוציה*: חלקים מלוח הזמנים הישראלי מוגדרים ברזולוציית דקה בלבד, ולכן זוגות תחנות עוקבות מסוימים
  נושאים חותמות זמן זהות ומניבים משך של אפס בדיוק. אנו מסירים אותם במקום לקבוע אותם לערך מזערי, מה
  שמטה כל median ששרד מעט **כלפי מעלה** ומסיר לחלוטין את הקטעים שבהם אף trip לא הפיק משך חיובי -
  באופן לא-פרופורציונלי הקישורים העירוניים הקצרים והצפופים, ולכן ההטיה אינה מתפרסת באופן אחיד על פני
  הרשת. סעיף 8 מדפיס את הספירות המדויקות וסעיף 9 מדפיס כמה קטעים משלב 02 אבדו; שני מספרים אלו הם
  המגבלה העיקרית של גרף זמני הנסיעה ויש לצטט אותם לצד כל תוצאה הנגזרת ממנו.

* **ההגעה והיציאה זהות בכל שורה ב-feed זה, ולכן אין זמן שהייה (dwell time).** "זמן הנסיעה" שלנו הוא אך
  ורק הפער המתוכנן בין עזיבת תחנה אחת להגעה לתחנה הבאה. המתנה בתחנה, מעבר בין קווים והמתנה לשירות הבא
  כולם בלתי נראים. זמן המסע של נוסע אמיתי ארוך אפוא בהחלט מכל מספר במחברת זו, והפער גדול ביותר בדיוק
  היכן שהמעברים חשובים ביותר - וזוהי הטיה נגד מוקדי תחבורה רב-אופניים (multimodal).

* **מתוכנן, לא ממומש.** GTFS הוא לוח זמנים. גודש, התקבצות (bunching) ואיחורים אינם בנתונים, ולכן אלו
  הזמנים שהמפעיל *מתכוון* אליהם, ולא הזמנים שהנוסע *חווה*.

* **השוואת המסלולים מכמתת ישירות את עלות ההנחה הישנה.** ה"זמן החציוני שאבד בגלל ניתוב לפי קפיצות"
  המודפס הוא המספר שיש לצטט: זהו מידת האיטיות של מסלול אופטימלי-בקפיצות ביחס למסלול אופטימלי-בזמן על
  אותה רשת, בדקות, על פני מדגם אקראי של זוגות מוצא-יעד. חלקם של הזוגות ששני מסלוליהם זהים לחלוטין הוא
  הסטטיסטיקה המשלימה - היכן שהוא גבוה, המחברות הקודמות צדקו במקרה.

* **בנוגע לדירוג ה-betweenness, קראו את טבלת ההסכמה בת שלוש השורות, ולא מספר בודד.** ההשוואה המבודדת
  את אפקט השקלול היא `travel-time vs hop control`, משום ששתי ההרצות השתמשו באותו `k` ובאותו seed ולכן
  באותם מקורות שנדגמו. השורה `hop control vs notebook 04` היא רצפת הרעש: שני אומדני betweenness מבוססי
  קפיצות הנבדלים זה מזה רק בדגימה. **שינוי בדירוג לפי זמן נסיעה מהווה ראיה לגבי השקלול רק אם הוא עולה
  על אותה רצפה.** אם חפיפת ה-top-50 בין זמן הנסיעה לספירת הקפיצות נמוכה באופן מהותי מן החפיפה של רצפת
  הרעש, אזי כל טענה קודמת בפרויקט זה מסוג "התחנה הקריטית ביותר" תלוית-שקלול וחייבת לשאת את ההסתייגות
  הזו; אם לא, המסקנה הכנה היא שב-`k = 200` איננו יכולים להפריד בין שני האפקטים, והתרופה היא
  `K_BETWEENNESS` גדול יותר, ולא טענה חזקה יותר.

* **ה-betweenness כאן מדגמי ולכן רועש.** עם `k = 200` מקורות מתוך כ-30k תחנות, רוב התחנות מקבלות ציון
  אפס בדיוק פשוט משום שאף מסלול קצר ביותר שנדגם לא חצה אותן. זו הסיבה שניתוח הזזות הדירוג מוגבל למאגר
  ה-top-`RANK_POOL`, ומדוע הזזות דירוג עבור תחנות בדירוג נמוך אינן מדווחות כלל - הן היו תוצר לוואי של
  שבירת שוויון.

* **המשקלים עודם היצע, לא ביקוש.** כמו בכל שלב קודם, הגרף מתאר את השירות שהמפעיל מפעיל, ולא את הנוסעים
  המשתמשים בו. זמן הנסיעה הופך את מודל ההיצע לריאליסטי בהרבה; הוא אינו הופך אותו למודל ביקוש. מחברת 21
  היא המקום שבו נכנס פרוקסי לביקוש.